# 08 — Chain pipeline: многошаговый lineage через intermediate tables

Демонстрация **многохопового** column lineage: цепочка из 3 шагов, где каждый промежуточный результат материализуется в отдельный Hive table. В Marquez UI у финального датасета можно идти upstream и видеть полный путь до сырых источников.

```
  raw_customers ─┐
                 ├──> tmp_orders_with_customer ──> tmp_country_metrics ──> mart_top_countries
  raw_orders ────┘     (join, denormalize)         (group by country)      (filter top-3)
```

> ⚠️ **`createOrReplaceTempView` в lineage НЕ появляется.** Temp view — это лишь имя для in-memory логического плана, OL-Spark не эмитит для него dataset events. Чтобы Marquez увидел промежуточный узел в графе, его нужно **физически материализовать** (`saveAsTable` / `insertInto`). Здесь все `tmp_*` — реальные Hive-таблицы.

Prerequisite: прогнан `00_setup.ipynb`. **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_08_chain_pipeline')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_08_chain_pipeline', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

# Чистая цепочка — дропаем старые tmp/mart если есть.
for t in ['tmp_orders_with_customer', 'tmp_country_metrics', 'mart_top_countries']:
    spark.sql(f'DROP TABLE IF EXISTS default.{t}')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:13:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:13:33 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:13:44 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:13:44 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:13:44 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_08_chain_pipeline


26/05/27 13:13:46 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:13:46 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:13:46 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:13:46 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


## Шаг 1: denormalize — `raw_customers + raw_orders` → `tmp_orders_with_customer`

Join клиентов с заказами, оставляем нужные колонки. Промежуточный результат — материализуем в Hive.

In [2]:
step1 = spark.sql('''
    SELECT
        o.order_id,
        o.amount,
        o.order_ts,
        c.id        AS customer_id,
        c.name      AS customer_name,
        c.country   AS country
    FROM default.raw_orders o
    INNER JOIN default.raw_customers c
        ON c.id = o.customer_id
''')
step1.write.mode('overwrite').saveAsTable('default.tmp_orders_with_customer')
print('step 1 rows:', spark.table('default.tmp_orders_with_customer').count())

26/05/27 13:13:47 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:13:47 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:13:47 WARN OutputStatisticsOutputDatasetFacetBuilder: No jobId found in context
26/05/27 13:13:47 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:13:47 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:13:47 ERROR ContextFactory: Query execution is null: can't emit event for executionId 1
26/05/27 13:13:47 WARN OutputStatisticsOutputDatasetFacetBuilder: No jobId found in context
26/05/27 13:13:47 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedId

step 1 rows: 500


## Шаг 2: aggregate — `tmp_orders_with_customer` → `tmp_country_metrics`

Группировка по стране. Входной датасет — результат шага 1. В Marquez `tmp_country_metrics` будет иметь column lineage не на `raw_*`, а на `tmp_orders_with_customer` (один шаг назад).

In [3]:
from pyspark.sql import functions as F

step2 = spark.sql('''
    SELECT
        country,
        count(*)              AS orders_count,
        count(DISTINCT customer_id) AS unique_customers,
        sum(amount)           AS revenue,
        avg(amount)           AS avg_check
    FROM default.tmp_orders_with_customer
    GROUP BY country
''')
step2.write.mode('overwrite').saveAsTable('default.tmp_country_metrics')
spark.table('default.tmp_country_metrics').orderBy(F.col('revenue').desc()).show(truncate=False)

+-------+------------+----------------+------------------+-----------------+
|country|orders_count|unique_customers|revenue           |avg_check        |
+-------+------------+----------------+------------------+-----------------+
|RU     |125         |25              |64487.63786179853 |515.9011028943883|
|US     |125         |25              |62105.7507133878  |496.8460057071024|
|FR     |125         |25              |61368.857552241105|490.9508604179288|
|DE     |125         |25              |59395.2885332553  |475.1623082660424|
+-------+------------+----------------+------------------+-----------------+



## Шаг 3: filter + project — `tmp_country_metrics` → `mart_top_countries`

Финальный mart: top-3 страны по revenue с добавлением расчётного поля. Column lineage:
- `mart_top_countries.country` ← `tmp_country_metrics.country` ← `tmp_orders_with_customer.country` ← `raw_customers.country` (3 хопа!)
- `mart_top_countries.revenue_per_customer` = `revenue / unique_customers` — двувходовая `TRANSFORMATION`.

> ℹ️ `ORDER BY` внутри `saveAsTable`-запроса сортирует данные **перед** записью, но Hive-таблица не хранит порядок строк — при последующем `SELECT * FROM mart_top_countries` строки могут вернуться в любом порядке. `LIMIT 3` всё равно даёт ровно 3 строки (топ-3 по revenue), просто не рассчитывай на их порядок в самой таблице.

In [4]:
step3 = spark.sql('''
    SELECT
        country,
        orders_count,
        unique_customers,
        revenue,
        revenue / unique_customers AS revenue_per_customer
    FROM default.tmp_country_metrics
    ORDER BY revenue DESC
    LIMIT 3
''')
step3.write.mode('overwrite').saveAsTable('default.mart_top_countries')
spark.table('default.mart_top_countries').show(truncate=False)

+-------+------------+----------------+------------------+--------------------+
|country|orders_count|unique_customers|revenue           |revenue_per_customer|
+-------+------------+----------------+------------------+--------------------+
|RU     |125         |25              |64487.63786179853 |2579.5055144719413  |
|US     |125         |25              |62105.7507133878  |2484.2300285355122  |
|FR     |125         |25              |61368.857552241105|2454.754302089644   |
+-------+------------+----------------+------------------+--------------------+



In [5]:
spark.stop()

26/05/27 13:13:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 11
26/05/27 13:13:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 11


## Что смотреть в Marquez

### Dataset-level граф
UI → datasets → `default.mart_top_countries` → tab **Lineage**. Должен быть виден граф:

```
raw_customers ─┐
               ├──> tmp_orders_with_customer ──> tmp_country_metrics ──> mart_top_countries
raw_orders ────┘
```

Dataset/job граф в Marquez **транзитивен**: можно сразу увидеть всю цепочку или запросить через API:

```bash
curl -s 'http://localhost:5000/api/v1/lineage?nodeId=dataset:hadoop-cluster:default.mart_top_countries&depth=10' \
  | jq '.graph[] | {id, type}'
```

### Column lineage — НЕ транзитивно
Column lineage в Marquez 0.47 хранится в `columnLineage` facet на каждой версии датасета и указывает только **на непосредственный upstream**. Чтобы пройти полный путь `mart_top_countries.country` ← ... ← `raw_customers.country`, нужно либо:

1. Кликать в UI: `mart_top_countries` → видим источник `tmp_country_metrics.country` → переход в `tmp_country_metrics` → видим `tmp_orders_with_customer.country` → переход → видим `raw_customers.country`.
2. Или собрать вручную через API:
   ```bash
   for D in mart_top_countries tmp_country_metrics tmp_orders_with_customer; do
     echo "=== $D ==="
     curl -s "http://localhost:5000/api/v1/namespaces/hadoop-cluster/datasets/default.$D/versions" \
       | jq '.versions[0].facets.columnLineage // "<no facet>"'
   done
   ```

### Три отдельных job'а в Marquez
Под одним appName `lineage_08_chain_pipeline` Marquez покажет **три** разных job'а (по одному на каждый `saveAsTable`) — это нормально: OL-Spark формирует job name из app + output table. Все три run'а имеют один `spark.app.id`, что можно найти в run-facets'ах.

## Чем НЕ является `createOrReplaceTempView`

Можно было бы сделать всю цепочку через temp views — синтаксически чище, никаких Hive-таблиц-помойки:

```python
step1.createOrReplaceTempView('v_orders_with_customer')
step2 = spark.sql('SELECT country, sum(amount) AS revenue FROM v_orders_with_customer GROUP BY country')
step2.createOrReplaceTempView('v_country_metrics')
# ...
step3.write.saveAsTable('default.mart_top_countries')  # <- только это попадёт в lineage
```

Но Marquez в этом случае увидит **только один job** `mart_top_countries`, с inputs `raw_customers` + `raw_orders` (Spark inlинит temp view в общий план). Промежуточные узлы графа пропадают.

Поэтому для целей наглядного lineage **материализация в Hive обязательна**.